# Decoding knobs, live: temperature, top-p, top-k, penalties

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 09 §9.1 · type: concept-lab*

**The promise:** you will reshape a real probability distribution with each sampling knob, watch the draw change, and learn to set these parameters deliberately per call type — without ever confusing *variety* with *truth*.

## 🧠 Why this matters

Chapter 8 ended with the model emitting a probability distribution over the next token: `" Paris"` 92%, `" the"` 3%, a long tail of everything else. **Decoding** is the policy for picking one token from that distribution — and it is pure engineering that happens *after* the model has done its thinking.

The mental model is one sentence: **decoding reshapes the distribution, then rolls a die on it.** `temperature` reshapes the whole curve (sharpen or flatten). `top_p` and `top_k` *truncate* — they cut the unreliable tail off before the roll. None of these knobs adds capability the model didn't already have; they only trade **consistency against variety**. A wrong fact at temperature 0 is still wrong, because hallucination lives in the distribution, not in the sampler.

## Objectives & prereqs

**By the end you can:**
- Apply a temperature scale to logits and re-softmax, and explain the effect on the draw.
- Implement nucleus (top-p) and top-k truncation and see exactly what gets cut.
- Justify a sampling-parameter choice for any call type (tool call vs. ideation).

**Prereqs:** Ch 8 (next-token prediction, the distribution) and its notebooks `learn/part-03-llm-substrate/08-*`. Read book §9.1 alongside this.

**Runs free & offline.** All distribution math is local NumPy. The one optional live cell needs `ANTHROPIC_API_KEY` and `COMPANION_MOCK=0`; leave the default and it costs nothing.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os

import numpy as np
from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default) keeps this notebook free, offline, and deterministic.
# Flip COMPANION_MOCK=0 in your .env only to run the single live comparison cell.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed everything stochastic so the dice rolls below are reproducible.
RNG = np.random.default_rng(7)

print("MOCK mode:" , MOCK, "— offline, no API key needed" if MOCK else "— LIVE, will call the API")

## A hand-made next-token distribution

We start fully offline. Here is a toy distribution over the next token after the prompt *"The capital of France is"* — the shape a real model would emit. We store the **logits** (the raw pre-softmax scores), because temperature operates on logits, not on probabilities.

In [ ]:
TOKENS = [" Paris", " the", " a", " located", " home", " one", " famous", " Lyon"]
# Logits chosen so a plain softmax gives ~92% to " Paris" — a confident, peaked distribution.
LOGITS = np.array([8.0, 4.5, 3.6, 3.2, 2.9, 2.4, 1.8, 1.2])


def softmax(logits, temperature=1.0):
    """Temperature-scaled softmax. temperature -> 0 approaches greedy (argmax)."""
    # Divide logits by T BEFORE softmax: small T sharpens, large T flattens.
    z = np.asarray(logits, dtype=float) / max(temperature, 1e-9)
    z = z - z.max()  # numerical stability
    e = np.exp(z)
    return e / e.sum()


def show(probs, label):
    print(label)
    for tok, p in sorted(zip(TOKENS, probs), key=lambda kv: -kv[1]):
        bar = "█" * int(round(p * 40))
        print(f"  {tok:>10}  {p:6.2%}  {bar}")
    print()


base = softmax(LOGITS, temperature=1.0)
show(base, "Base distribution (temperature = 1.0):")

## 🔮 Predict

Before you run the next cell: we will re-softmax the **same logits** at temperature **0.3** (cold) and **1.6** (hot).

- At **T = 0.3**, does `" Paris"` get *more* or *less* of the probability mass?
- At **T = 1.6**, what happens to the long tail (`" Lyon"`, `" famous"`)?

Write your guess down, then run it.

In [ ]:
show(softmax(LOGITS, temperature=0.3), "Cold (temperature = 0.3) — sharpened:")
show(softmax(LOGITS, temperature=1.6), "Hot  (temperature = 1.6) — flattened:")

**What you just saw.** Cold temperature *sharpens*: `" Paris"` swallows almost all the mass, so sampling becomes near-deterministic (this is why low temperature approaches greedy decoding). Hot temperature *flattens*: the tail tokens get real probability, so a draw is more likely to wander somewhere surprising. The logits — the model's actual judgement — never moved. Temperature only changed how adventurously we read them.

## Truncation: top-p (nucleus) and top-k

Temperature reshapes the *entire* curve, tail included. Truncation throws the tail away *before* the roll. **Top-k** keeps the `k` most-likely tokens; **top-p** (nucleus) keeps the smallest set whose probabilities sum to `p`, then renormalizes. Let's implement both and print exactly which tokens survive.

In [ ]:
def top_k(probs, k):
    """Keep the k most-likely tokens; zero the rest; renormalize."""
    probs = np.asarray(probs, dtype=float)
    keep = np.argsort(probs)[::-1][:k]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()


def top_p(probs, p):
    """Nucleus sampling: keep the smallest set of tokens whose cumulative prob >= p."""
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    # Include tokens up to and including the one that crosses the p threshold.
    cutoff = np.searchsorted(cum, p) + 1
    keep = order[:cutoff]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()


show(top_k(base, k=3), "top-k = 3 (everything past the top 3 is cut):")
show(top_p(base, p=0.95), "top-p = 0.95 (smallest nucleus reaching 95% cumulative):")

**What you just saw.** Top-k is a blunt count: exactly three tokens survive regardless of how the mass is distributed. Top-p is adaptive: when one token already holds 92%, the nucleus is *tiny* (often just `" Paris"` plus one or two others) — because the distribution is confident. On a flat, uncertain distribution the same `p=0.95` would keep many more tokens. That adaptivity is why top-p is usually the tail-cutter of choice.

## ⚠️ Pitfall: stacking temperature *and* top-p aggressively

These knobs interact. Crank temperature up to flatten the curve, *then* apply a tight top-p, and the two fight: temperature inflates tail tokens, top-p tries to cut them, and the result is hard to reason about and unstable across calls. The book's guidance: **tune one or the other, not both aggressively.** Watch what high temperature does to the surviving nucleus.

In [ ]:
hot = softmax(LOGITS, temperature=1.6)
print("Nucleus size (number of tokens kept by top-p = 0.9):")
print("  at temperature 1.0:", int((top_p(base, 0.9) > 0).sum()), "tokens")
print("  at temperature 1.6:", int((top_p(hot, 0.9) > 0).sum()), "tokens")
print("\nSame p, very different behavior — because temperature already moved the mass.")

## 🔮 Predict: does sampling change a *wrong* fact?

Suppose the model is confidently wrong — its distribution puts most mass on an incorrect token. We will *sample* that distribution many times at temperature 0 and at temperature 1.0.

**Predict:** does raising temperature make the answer more likely to be *correct*? Or does it only change how *consistent* the output is? Decide before running.

In [ ]:
# A confidently-WRONG distribution: the model is 88% sure the answer is " Berlin".
wrong_tokens = [" Berlin", " Paris", " Madrid", " Rome"]
wrong_probs = np.array([0.88, 0.07, 0.03, 0.02])
TRUTH = " Paris"


def sample_once(tokens, probs, temperature):
    if temperature <= 1e-9:  # temperature 0 == greedy == always the argmax
        return tokens[int(np.argmax(probs))]
    # Re-temperature the *probabilities* via their implied logits, then draw.
    logits = np.log(np.asarray(probs, dtype=float) + 1e-12)
    p = softmax(logits, temperature)
    return tokens[RNG.choice(len(tokens), p=p)]


for T in (0.0, 1.0):
    draws = [sample_once(wrong_tokens, wrong_probs, T) for _ in range(2000)]
    correct = sum(d == TRUTH for d in draws) / len(draws)
    most_common = max(set(draws), key=draws.count)
    print(f"T={T}:  picks {most_common!r} most often;  correct ({TRUTH!r}) only {correct:.1%} of draws")

**What you just saw.** Temperature 0 *always* returns the wrong token (the argmax is wrong). Temperature 1.0 stumbles onto the correct answer only a few percent of the time — and *less* consistently. Sampling changed **consistency, not correctness**. The hallucination is baked into the distribution; no knob on the sampler can fix a model that is sure of the wrong thing. Turning temperature down buys you *repeatability*, never *truth*.

## Optional live cell: variety at temp 0.0 / 0.7 / 1.0

With `COMPANION_MOCK=0` and an `ANTHROPIC_API_KEY`, this sends the same creative prompt at three temperatures so you can eyeball the variety. **Cost:** three short generations (a few cents at most). In `MOCK=1` it replays canned responses that illustrate the same point for free.

> Note: as of early 2026 the latest Opus-tier models steer behavior through prompting and **do not accept `temperature`** (the request 400s). This cell therefore targets `claude-sonnet-4-6`, which still exposes the decoding knobs — exactly the kind of per-call-type model choice this chapter is teaching.

In [ ]:
PROMPT = "In one short sentence, describe a city at dawn."
MODEL = "claude-sonnet-4-6"  # still accepts temperature; see note above

# Canned, realistic outputs so MOCK mode shows the SAME teaching point offline.
MOCK_REPLIES = {
    0.0: ["The city wakes slowly under a pale grey sky.",
          "The city wakes slowly under a pale grey sky.",
          "The city wakes slowly under a pale grey sky."],
    0.7: ["Soft amber light spills over quiet rooftops as the city stirs.",
          "The city yawns awake beneath a wash of rose and gold.",
          "Streetlights fade as dawn gilds the empty avenues."],
    1.0: ["Mist curls between towers while the first trams rattle to life.",
          "A copper sun pries the city open, one shuttered window at a time.",
          "Dawn arrives like a rumor — gulls, diesel, and a sky bruised pink."],
}


def generate(temperature):
    """Return three completions at the given temperature (mock or live)."""
    if MOCK:
        return MOCK_REPLIES[temperature]
    import anthropic  # imported only on the live path

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    outs = []
    for _ in range(3):
        msg = client.messages.create(
            model=MODEL,
            max_tokens=40,
            temperature=temperature,
            messages=[{"role": "user", "content": PROMPT}],
        )
        outs.append("".join(b.text for b in msg.content if b.type == "text").strip())
    return outs


for T in (0.0, 0.7, 1.0):
    print(f"--- temperature = {T} ---")
    for line in generate(T):
        print("  •", line)
    print()

## 🎯 Senior lens

Agentic systems live at **low temperature**. When the model decides which tool to call, or emits JSON for a downstream parser, variety is a *liability* — you want the same input to produce the same structured output. Raise temperature only for steps that are *explicitly* creative (drafting copy, brainstorming options), and even then within a tested range.

Treat sampling parameters as **part of the prompt's contract** (Ch 10): a prompt tuned at temperature 0.2 is a *different artifact* at 1.0, and an eval run at one setting tells you nothing about the other. Store the parameters with the prompt, version them together, and run evals at the setting you ship. The capstone's LLM client encodes this: low temperature by default for agent and extraction calls, raised deliberately and locally where creativity is the point.

## Recap

- **Decoding = reshape, then roll.** Temperature reshapes the whole curve; top-p / top-k truncate the tail before the draw.
- **Temperature trades consistency for variety** — never truth. A confident wrong answer stays wrong at every temperature.
- **Top-p is adaptive**, top-k is a blunt count; prefer top-p as the tail-cutter and tune *one* knob, not both aggressively.
- **Low temperature for tool calls, extraction, and JSON;** raise it only for explicitly creative steps.
- **Parameters are part of the prompt contract** — versioned together, evaluated at the setting you ship.

## Exercises

Each one *changes a knob* and asks you to predict before running.

1. **Find the greedy threshold.** Sweep temperature from 1.0 down to 0.05 on `LOGITS` and report the temperature at which `" Paris"` first exceeds 99%. Predict whether it's above or below 0.3 first.
2. **Top-p on a flat distribution.** Build a near-uniform distribution over the 8 tokens and apply `top_p(probs, 0.95)`. Predict how many tokens survive *now* versus on the peaked `base`, then check.
3. **Penalty intuition.** Add a crude frequency penalty: subtract a constant from the logit of any token already "generated", re-softmax, and observe how repetition is discouraged. Predict what an *aggressive* penalty does to legitimately repeated tokens (think code, or a name).

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **Next notebook:** [`09-02-determinism-seeds-and-replay.ipynb`](09-02-determinism-seeds-and-replay.ipynb) — why temperature 0 still isn't byte-for-byte deterministic, and how to engineer reproducibility anyway.
- **Book:** §9.1 (decoding) and the sampling-parameters contract idea continued in §10.
- **Blueprint this feeds:** [`blueprints/llm-gateway/`](../../blueprints/llm-gateway/) — where these sampling defaults become a single configurable door (built from Ch 11).
- **Capstone:** sets the defaults the capstone `llm/` client encodes — low temperature for agent and extraction calls.